# H2 — Kiểm tra chất lượng token cache

Notebook này **không chạy ResNet/encoder và không huấn luyện model**. Nó đọc hai manifest ADNI hiện có, tạo cache `.pt` giả theo schema Neuro-JEPA `[576,1024]`, rồi chứng minh `scripts/check_token_cache.py` phát hiện được NaN/Inf, sai shape, cache thiếu, duplicate scan và subject leakage.

Dữ liệu giả chỉ nằm trong `outputs/h2_cache_sanity_demo/` (đã gitignore). Khi cache thật sẵn sàng, thay mini-manifest/cache directory bằng mapping chính thức từ anh Long.

In [ ]:
from __future__ import annotations

import csv
import json
import shutil
import sys
from collections import Counter, defaultdict
from pathlib import Path

import torch

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'scripts/check_token_cache.py').is_file():
    raise RuntimeError('Run this notebook from the NeuRL repository or notebooks/ directory')
sys.path.insert(0, str(REPO_ROOT))

from scripts.check_token_cache import audit_cache

MANIFESTS = [
    REPO_ROOT / 'data/manifests/2026.04.24_ADNI.csv',
    REPO_ROOT / 'data/manifests/2026.04.28_ADNI.csv',
]
for path in MANIFESTS:
    assert path.is_file(), f'Missing manifest: {path}'

DEMO_ROOT = REPO_ROOT / 'outputs/h2_cache_sanity_demo'
if DEMO_ROOT.exists():
    shutil.rmtree(DEMO_ROOT)
DEMO_ROOT.mkdir(parents=True)
print('Demo output:', DEMO_ROOT)

## 1. Kiểm tra hai manifest thật

Hai file này là manifest lâm sàng + ảnh, chưa phải cache manifest: hiện chưa có cột `cache_file` hoặc `split`, và một số hàng không có `IMAGEUID/file_path`. Do đó không được giả định mọi hàng đều phải có `.pt`. Trước khi chạy cache thật, nhóm phải chốt cohort MRI đủ điều kiện, split và quy tắc ánh xạ mỗi hàng sang đúng một file cache.

In [ ]:
REQUIRED_SOURCE_COLUMNS = ('PTID', 'VISCODE', 'IMAGEUID', 'EXAMDATE', 'file_path')

def profile_manifest(path: Path) -> dict:
    subjects, visits, scans = set(), set(), Counter()
    missing = Counter()
    row_count = 0
    with path.open(encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle)
        absent = [name for name in REQUIRED_SOURCE_COLUMNS if name not in (reader.fieldnames or [])]
        if absent:
            raise ValueError(f'{path.name} is missing columns: {absent}')
        for row in reader:
            row_count += 1
            values = {name: (row.get(name) or '').strip() for name in REQUIRED_SOURCE_COLUMNS}
            for name, value in values.items():
                if not value:
                    missing[name] += 1
            if values['PTID']:
                subjects.add(values['PTID'])
            visits.add((values['PTID'], values['VISCODE']))
            scans[(values['PTID'], values['VISCODE'], values['IMAGEUID'])] += 1
    return {
        'manifest': path.name,
        'rows': row_count,
        'subjects': len(subjects),
        'unique_visits': len(visits),
        'unique_scan_keys': len(scans),
        'duplicate_scan_key_groups': sum(count > 1 for count in scans.values()),
        'missing_IMAGEUID': missing['IMAGEUID'],
        'missing_file_path': missing['file_path'],
    }

profiles = [profile_manifest(path) for path in MANIFESTS]
print(json.dumps(profiles, indent=2, ensure_ascii=False))
assert all(item['rows'] > 0 for item in profiles)

## 2. Tạo mini-manifest và cache giả chất lượng cao

Lấy các hàng có MRI từ manifest mới hơn. Bộ mẫu cố ý chứa ít nhất hai lần khám của cùng PTID để kiểm tra longitudinal visit hợp lệ. Khóa scan dùng `PTID + VISCODE + IMAGEUID`; chỉ PTID giống nhau chưa phải duplicate.

Mỗi cache giả có đúng schema Neuro-JEPA: `embeddings [576,1024]`, `coords [576,3]`, `grid_shape [8,9,8]`, metadata và `brain_fraction`. Đây chỉ là tensor giả phục vụ validation plumbing, không phải embedding y khoa.

In [ ]:
def load_eligible_rows(path: Path) -> list[dict[str, str]]:
    keep = ('PTID', 'VISCODE', 'IMAGEUID', 'EXAMDATE', 'file_path')
    rows = []
    with path.open(encoding='utf-8-sig', newline='') as handle:
        for raw in csv.DictReader(handle):
            row = {name: (raw.get(name) or '').strip() for name in keep}
            if all(row[name] for name in ('PTID', 'VISCODE', 'IMAGEUID', 'file_path')):
                rows.append(row)
    return rows

def normalized_image_id(value: str) -> str:
    return value[:-2] if value.endswith('.0') else value

def cache_name(row: dict[str, str]) -> str:
    return f"{row['PTID']}__{row['VISCODE']}__I{normalized_image_id(row['IMAGEUID'])}.pt"

def choose_demo_rows(rows: list[dict[str, str]]) -> list[dict[str, str]]:
    by_subject = defaultdict(list)
    for row in rows:
        by_subject[row['PTID']].append(row)
    longitudinal = next((items for items in by_subject.values() if len(items) >= 2), None)
    if longitudinal is None:
        raise RuntimeError('No longitudinal subject found in the eligible manifest rows')
    chosen = longitudinal[:2]
    used = {chosen[0]['PTID']}
    for row in rows:
        if row['PTID'] not in used:
            chosen.append(row)
            used.add(row['PTID'])
        if len(chosen) == 5:
            break
    assert len(chosen) == 5
    return chosen

def write_audit_manifest(path: Path, rows: list[dict[str, str]]) -> None:
    columns = ('PTID', 'VISCODE', 'IMAGEUID', 'EXAMDATE', 'file_path', 'split', 'cache_file')
    with path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows({name: row.get(name, '') for name in columns} for row in rows)

GRID_SHAPE = (8, 9, 8)
COORDS = torch.stack(
    torch.meshgrid(*(torch.arange(size) for size in GRID_SHAPE), indexing='ij'),
    dim=-1,
).reshape(-1, 3).short()

def make_fake_cache(row: dict[str, str], seed: int) -> dict:
    generator = torch.Generator().manual_seed(seed)
    embeddings = torch.randn(576, 1024, generator=generator).half()
    return {
        'embeddings': embeddings,
        'coords': COORDS.clone(),
        'token_count': 576,
        'embedding_dim': 1024,
        'grid_shape': torch.tensor(GRID_SHAPE, dtype=torch.int32),
        'brain_fraction': torch.rand(576, generator=generator),
        'scan_path': row['file_path'],
        'metadata': json.dumps({'encoder': 'fake-neurojepa-for-H2-only'}),
    }

eligible = load_eligible_rows(MANIFESTS[-1])
selected = choose_demo_rows(eligible)
splits = ('train', 'train', 'train', 'validation', 'test')
clean_rows = [{**row, 'split': split_name, 'cache_file': cache_name(row)} for row, split_name in zip(selected, splits)]
print('Eligible MRI rows in latest manifest:', len(eligible))
print('Selected identities:', [(r['PTID'], r['VISCODE'], normalized_image_id(r['IMAGEUID'])) for r in clean_rows])

## 3. Kịch bản sạch phải PASS

Hai visit đầu cùng PTID và cùng split. Checker phải coi đây là theo dõi dọc hợp lệ, không phải duplicate hoặc leakage.

In [ ]:
clean_root = DEMO_ROOT / 'clean'
clean_cache_dir = clean_root / 'tokens'
clean_cache_dir.mkdir(parents=True)
clean_manifest = clean_root / 'manifest_with_expected_cache.csv'
write_audit_manifest(clean_manifest, clean_rows)
for index, row in enumerate(clean_rows):
    torch.save(make_fake_cache(row, seed=100 + index), clean_cache_dir / row['cache_file'])

clean_report = audit_cache(
    manifest=clean_manifest,
    cache_dir=clean_cache_dir,
    output=clean_root / 'cache_check.json',
    key_columns=('PTID', 'VISCODE', 'IMAGEUID'),
)
print(json.dumps(clean_report['summary'], indent=2))
assert clean_report['status'] == 'PASS'
assert clean_report['summary']['matched_cache_files'] == len(clean_rows)
assert clean_report['summary']['subject_split_leakage'] == 0
assert clean_report['summary']['duplicate_manifest_identities'] == 0
print('CLEAN CASE: PASS')

## 4. Kịch bản lỗi phải FAIL đúng nguyên nhân

Bộ lỗi gồm: NaN + Inf, `coords` thiếu một hàng, một cache bị thiếu hoàn toàn, một identity/cache reference lặp trong manifest, một file ngoài manifest dùng lại cùng `scan_path`, và cùng PTID bị đặt vào train lẫn test.

In [ ]:
broken_root = DEMO_ROOT / 'broken'
broken_cache_dir = broken_root / 'tokens'
broken_cache_dir.mkdir(parents=True)
broken_rows = [dict(row) for row in clean_rows]
broken_rows[1]['split'] = 'test'  # same PTID now leaks across train/test
broken_rows.append(dict(broken_rows[-1]))  # duplicate scan identity + cache reference
broken_manifest = broken_root / 'manifest_with_expected_cache.csv'
write_audit_manifest(broken_manifest, broken_rows)

torch.save(make_fake_cache(broken_rows[0], 200), broken_cache_dir / broken_rows[0]['cache_file'])
torch.save(make_fake_cache(broken_rows[1], 201), broken_cache_dir / broken_rows[1]['cache_file'])
nan_cache = make_fake_cache(broken_rows[2], 202)
nan_cache['embeddings'][0, 0] = float('nan')
nan_cache['embeddings'][0, 1] = float('inf')
torch.save(nan_cache, broken_cache_dir / broken_rows[2]['cache_file'])
wrong_shape_cache = make_fake_cache(broken_rows[3], 203)
wrong_shape_cache['coords'] = wrong_shape_cache['coords'][:-1]
torch.save(wrong_shape_cache, broken_cache_dir / broken_rows[3]['cache_file'])
# broken_rows[4] is deliberately missing
torch.save(make_fake_cache(broken_rows[0], 204), broken_cache_dir / 'unexpected_duplicate_source.pt')

broken_report = audit_cache(
    manifest=broken_manifest,
    cache_dir=broken_cache_dir,
    output=broken_root / 'cache_check.json',
    key_columns=('PTID', 'VISCODE', 'IMAGEUID'),
)
detected_codes = {item['code'] for item in broken_report['issues']}
expected_codes = {
    'NONFINITE_EMBEDDINGS',
    'TOKEN_COORD_COUNT_MISMATCH',
    'MISSING_CACHE',
    'UNEXPECTED_CACHE',
    'DUPLICATE_MANIFEST_IDENTITY',
    'DUPLICATE_CACHE_REFERENCE',
    'DUPLICATE_SOURCE_SCAN',
    'SUBJECT_SPLIT_LEAKAGE',
}
print(json.dumps(broken_report['summary'], indent=2))
print('Detected issue codes:', sorted(detected_codes))
assert broken_report['status'] == 'FAIL'
assert expected_codes <= detected_codes, expected_codes - detected_codes
assert broken_report['missing_subjects'] == [broken_rows[4]['PTID']]
assert (broken_root / 'cache_check.json').is_file()
assert (broken_root / 'cache_check_details.csv').is_file()
print('BROKEN CASE: EXPECTED FAIL — all target error classes detected')

## 5. Cách chạy khi anh Long giao cache thật

Hai manifest nguồn hiện tại **không có** `cache_file`/`split` và có hàng không gắn MRI. Trước tiên phải tạo một expected-cache manifest từ cohort đã chốt, giữ ít nhất `PTID`, `VISCODE`, `IMAGEUID`, thêm `split` và `cache_file` đúng theo quy ước tên của Long. Không tự đoán mapping tên file.

Sau đó chạy từ repo root:

```bash
PYTHONPATH=. .venv/bin/python scripts/check_token_cache.py \
  --cache-dir /path/to/long/adni/tokens \
  --manifest /path/to/manifest_with_expected_cache.csv \
  --key-columns PTID VISCODE IMAGEUID \
  --output reports/adni_cache_check.json
```

Exit code `0` = PASS, `1` = dữ liệu FAIL, `2` = cấu hình/đường dẫn sai. JSON chứa summary + toàn bộ issue; CSV chứa kết quả từng file. Nếu Long lưu nested folders thì thêm `--recursive`.